# 第 4 课：经验记忆的结构与构建

目标：理解一条经验为什么保存历史、未来、特征、意图、策略和置信度。

In [ ]:
import sys
from pathlib import Path

# 同时兼容：从仓库根目录启动 Jupyter，或从 notebooks/ 目录启动。
search_starts = [Path.cwd(), *Path.cwd().parents]
repo_root = next((path for path in search_starts if (path / "pyproject.toml").exists()), None)
if repo_root is None:
    raise RuntimeError("没有找到 pyproject.toml；请从仓库目录启动 Jupyter。")

src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

print(f"仓库根目录: {repo_root}")
print(f"Python: {sys.version.split()[0]}")

## 4.1 从训练窗口建立记忆

核心源码：[memory.py](../src/memcast_uav/memory.py)  
文字讲解：[04_memory.md](../tutorial/04_memory.md)

In [ ]:
from memcast_uav.data import make_synthetic_flight, make_train_test_windows
from memcast_uav.memory import build_memory

flight = make_synthetic_flight()
train, _ = make_train_test_windows(flight, split_index=504)
memory = build_memory(train, limit=5)
print("记忆条目数:", len(memory.entries))

## 4.2 分块查看一条经验

In [ ]:
entry = memory.entries[0]
print("ID:", entry.entry_id)
print("history:", entry.history.shape)
print("future:", entry.future.shape)
print("features:", entry.features.shape)
print("intent:", entry.intent)
print("strategy:", entry.strategy)
print("confidence:", entry.confidence)

## 4.3 用相对未来位移迁移机动模式

In [ ]:
displacement = entry.future_displacement
hypothetical_current_position = entry.history[-1] + [1000.0, -200.0, 10.0]
reused_candidate = hypothetical_current_position + displacement

print("未来相对位移末点:", displacement[-1].round(3).tolist())
print("迁移后候选末点:", reused_candidate[-1].round(3).tolist())

## 4.4 分块练习：设计环境上下文

先在下面字典中填写预测时已经可见的上下文。不要读取预测区间之后的信息。

In [ ]:
context_design = {
    "wind_enu_mps": [0.0, 0.0, 0.0],
    "obstacle_density": 0.0,
    "weather": "clear",
}
context_design

TODO：

1. 连续上下文如何标准化？
2. 离散天气如何计算相似度？
3. 哪些字段在预测时不可见，可能导致泄漏？

## 4.5 本课验收

In [ ]:
assert len({item.entry_id for item in memory.entries}) == len(memory.entries)
assert entry.future_displacement.shape == entry.future.shape

import subprocess

completed = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests/test_retrieval.py",
        "tests/test_pipeline.py",
        "-q",
    ],
    cwd=repo_root,
    check=True,
    text=True,
    capture_output=True,
)
print(completed.stdout)

[← 第 3 课](03_retrieval.ipynb) · [教程目录](README.md) · [下一课：物理约束反思 →](05_reflection.ipynb)